API validation. Checks the exclusion criteria against the source parquet tables: that the cell lines removed are exactly those above threshold, that combining criteria removes the union, and that similarity fails cleanly for cell lines with no expression profile. Requires the API running on localhost:8000.

In [1]:
import requests
import pandas as pd

API = "http://localhost:8000"
pdir = "../../data/parquet"

r = requests.get(f"{API}/health")
print(r.json())

{'status': 'healthy'}


In [2]:
import os
print(sorted(os.listdir(pdir)))

['.ipynb_checkpoints', 'dim_cell_line_parents.parquet', 'dim_cell_lines.parquet', 'fact_expression_depmap.parquet', 'fact_expression_hpa.parquet', 'fact_fusions.parquet', 'fact_metabolomics.parquet', 'fact_mirna.parquet', 'fact_mutations.parquet', 'fact_proteomics.parquet', 'fact_signatures.parquet', 'geo.parquet']


In [3]:
import requests
import pandas as pd

API = "http://localhost:8000"
pdir = "../../data/parquet"

def rank(**kwargs):
    body = {"genes": [{"hugo": "EGFR", "direction": "high"}], "top_n": 200}
    body.update(kwargs)
    return requests.post(f"{API}/api/rank", json=body).json()

# which cell lines should be excluded, straight from the source table
metab = pd.read_parquet(f"{pdir}/fact_metabolomics.parquet")
lactate_high = set(metab[(metab["metabolite"] == "lactate") & (metab["value"] > 6.0)]["ach_id"])
print("above threshold in source:", len(lactate_high))

base = {r["ach_id"] for r in rank()["results"]}
filt = {r["ach_id"] for r in rank(exclude_metabolite="lactate", metabolite_threshold=6.0)["results"]}

removed = base - filt
print("removed by the filter:", len(removed))
print("all removed were above threshold:", removed <= lactate_high)
print("wrongly removed:", removed - lactate_high)
print("survivors that should have gone:", (filt & lactate_high))

above threshold in source: 224
removed by the filter: 62
all removed were above threshold: True
wrongly removed: set()
survivors that should have gone: set()


In [4]:
# miRNA exclusion: the same check against the source table
mirna = pd.read_parquet(f"{pdir}/fact_mirna.parquet")
mir21_high = set(mirna[(mirna["mirna_id"] == "hsa-miR-21") & (mirna["value"] > 7400)]["ach_id"])
print("above threshold in source:", len(mir21_high))

filt = {r["ach_id"] for r in rank(exclude_mirna="hsa-miR-21", mirna_threshold=7400)["results"]}
removed = base - filt

print("removed by the filter:", len(removed))
print("wrongly removed:", removed - mir21_high)
print("survivors that should have gone:", filt & mir21_high)

above threshold in source: 237
removed by the filter: 49
wrongly removed: set()
survivors that should have gone: set()


In [13]:
# instability exclusion: signatures uses two thresholds combined with OR
sig = pd.read_parquet(f"{pdir}/fact_signatures.parquet")

MSI_MAX, CIN_MAX = 3.0, 0.7
unstable = set(sig[(sig["MSIScore"] > MSI_MAX) | (sig["CIN"] > CIN_MAX)]["ach_id"])
print("above threshold in source:", len(unstable))

filt = {r["ach_id"] for r in rank(msi_max=MSI_MAX, cin_max=CIN_MAX)["results"]}
removed = base - filt

print("removed by the filter:", len(removed))
print("wrongly removed:", removed - unstable)
print("survivors that should have gone:", filt & unstable)

above threshold in source: 729
removed by the filter: 74
wrongly removed: set()
survivors that should have gone: set()


In [14]:
# two criteria together should remove the union of both, not just one
MSI_MAX, CIN_MAX = 3.0, 0.7

unstable = set(sig[(sig["MSIScore"] > MSI_MAX) | (sig["CIN"] > CIN_MAX)]["ach_id"])
lactate_high = set(metab[(metab["metabolite"] == "lactate") & (metab["value"] > 6.0)]["ach_id"])

both = {r["ach_id"] for r in rank(msi_max=MSI_MAX, cin_max=CIN_MAX,
                                   exclude_metabolite="lactate",
                                   metabolite_threshold=6.0)["results"]}
removed = base - both

print("removed by both filters:", len(removed))
print("wrongly removed:", removed - (unstable | lactate_high))
print("survivors that should have gone:", both & (unstable | lactate_high))


removed by both filters: 115
wrongly removed: set()
survivors that should have gone: set()


In [15]:
# many cell lines have no depmap expression, so similarity should fail cleanly
dim = pd.read_parquet(f"{pdir}/dim_cell_lines.parquet")
depmap_ids = set(pd.read_parquet(f"{pdir}/fact_expression_depmap.parquet",
                                 columns=["ACH_ID"])["ACH_ID"].dropna().unique())
missing = sorted(set(dim["ach_id"]) - depmap_ids)
print("cell lines without expression:", len(missing))

r = requests.get(f"{API}/api/celllines/{missing[0]}/similar")
print(missing[0], "->", r.status_code, r.json())

cell lines without expression: 741
ACH-000047 -> 404 {'detail': "No expression profile for 'ACH-000047'"}
